In [23]:
# All classes have a method called __def__ init to assign values to the attributes in the brackets but only for attribute initialisation
#
#Have to use self attribute to ensure attributes are saved outside of the __def__ init constructor, otherwise the information would be lost outside of the object
import pandas as pd
import numpy as np

class PowerPlant:
    def __init__ (self, plant_name, country, capacity_mw, prim_fuel_type, commission_year, lat, lon, gen_data):

        self.plant_name = plant_name
        self.country = country
        self.capacity_mw = capacity_mw
        self.prim_fuel_type = prim_fuel_type
        self.commission_year = commission_year
        self.lat = lat
        self.lon = lon
        self.gen_data = gen_data

    def calculate_environmental_impact(self):
                raise NotImplementedError("Further information is needed. Please refer to subclasses")
    
    def __str__(self):
        return f"{self.plant_name} ({self.country}) - {self.capacity_mw} MW, {self.prim_fuel_type}"

    def __repr__(self):
        return (f"PowerPlant(plant_name={self.plant_name!r}, country={self.country!r}, "
            f"capacity_mw={self.capacity_mw!r}, prim_fuel_type={self.prim_fuel_type!r}, "
            f"commission_year={self.commission_year!r})")
    
    # Factory method
    @classmethod
    def create_plant_objects(cls, row):
            plant_type = row['type']

            if plant_type == 'RenewablePlant':
                estimated_generation = row['estimated_generation_gwh']
                capacity_mw = row['capacity_mw']
                intermittency_factor = estimated_generation / (capacity_mw * 8760)
                storage_capacity = capacity_mw
                gen_data = row[['generation_gwh_2013', 'generation_gwh_2014', 'generation_gwh_2015', 'generation_gwh_2016']]
                return RenewablePlant(row['name'], row['country_long'], row['capacity_mw'], intermittency_factor, storage_capacity, row['fuel1'], row['fuel1'], gen_data)
            elif plant_type == 'FossilFuelPlant':
                estimated_generation = row['estimated_generation_gwh']
                capacity_mw = row['capacity_mw']
                emissions_rate = estimated_generation * capacity_mw
                gen_data = row[['generation_gwh_2013', 'generation_gwh_2014', 'generation_gwh_2015', 'generation_gwh_2016']]
                return FossilFuelPlant(row['name'], row['country_long'], row['capacity_mw'], row['fuel1'], row['commissioning_year'], row['latitude'], row['longitude'], gen_data, None, None, row['fuel1'])            
            elif plant_type == 'NuclearPlant':
                gen_data = row[['generation_gwh_2013', 'generation_gwh_2014', 'generation_gwh_2015', 'generation_gwh_2016']]
                return NuclearPlant(row['name'], row['country_long'], row['capacity_mw'], row['fuel1'], row['commissioning_year'], row['latitude'], row['longitude'], gen_data, None, None)             
    
    #getters = lets you access an attribute @property
    #setters = lets you set an attribute's property @attribute.setter
    
    #Ensuring that a value error is returned if the capacity number is not equal to or above zero 
    @property #creates a getter 
    def capacity_mw(self):
        return self._capacity_mw

    @capacity_mw.setter
    def capacity_mw(self, pos_value):
        if not pos_value >= 0:
            raise ValueError("Please enter a number above zero")
        self._capacity_mw = pos_value


    #Ensuring that a value error is returned if the latitude  is not within the appropriate bounds
    @property
    def lat(self):
        return self._lat
    
    @lat.setter
    def lat(self, correct_coord_lat):
        if correct_coord_lat is not None and not pd.isna(correct_coord_lat) and abs(correct_coord_lat) > 90:
            raise ValueError("Please enter a latitude between -90 and +90")
        self._lat = correct_coord_lat

    #Ensuring that a value error is returned if the longitude  is not within the appropriate bounds
    @property
    def lon(self):
        return self._lon
    
    @lon.setter
    def lon(self, correct_coord_lon):
        if correct_coord_lon is not None and not pd.isna(correct_coord_lon) and abs(correct_coord_lon) > 180:
            raise ValueError("Please enter a longitude between -180 and +180")
        self._lon = correct_coord_lon
    
    @property 
    def country(self):
        return self._country
    
    @country.setter
    def country(self,  string_sure):
        if not isinstance(string_sure, str):
            raise TypeError("Please enter the country name using letters")
        self._country = string_sure


    #Include method to calculate capacity factor 
    def capacity_factor(self, actual_gen, potential_gen, total_hours):
        self.actual_gen = actual_gen
        self.potential_gen = potential_gen
        self.total_hours = total_hours

        #Actual calculation within method
        capacity_factor_calc = self.actual_gen / self.potential_gen * self.total_hours

        print(f"The capacity factor is {capacity_factor_calc} %")

        # Divide the actual energy produced (in MWh or kWh) by the maximum potential energy output *  hours in the period)

    #We use 'PowerPlant' in brackets to denote the class that the subclass is inheriting from

    #solar, wind, hydro
class RenewablePlant(PowerPlant):
    def __init__ (self, plant_name, country, capacity_mw, intermittency_factor, storage_capacity, type, prim_fuel_type, gen_data):
        
        super().__init__(plant_name, country, capacity_mw, prim_fuel_type, commission_year=None, lat=None, lon=None, gen_data=gen_data)
        self.intermittency_factor = intermittency_factor
        self.storage_capacity = storage_capacity
        self.type = type # solar, wind, hydro 
        self.gen_data = gen_data
        self.prim_fuel_type = prim_fuel_type

        #Polymorphism allows the same method to behave differently depending on the object that it belongs to 
    def calculate_environmental_impact(self):
            if self.type == "Solar":
                return self.capacity_mw * 0.01
            elif self.type == "Wind":
                return self.capacity_mw * 0.03
            elif self.type == "Hydro":
                return self.capacity_mw * 0.05
    
# coal, gas, oil 
class FossilFuelPlant(PowerPlant):
    def __init__(self, plant_name, country, capacity_mw, prim_fuel_type, commission_year, lat, lon, gen_data, emission_rate, fuel_efficiency, type):

        super().__init__(plant_name, country, capacity_mw, prim_fuel_type, commission_year, lat, lon, gen_data)

        self.emission_rate = emission_rate
        self.fuel_efficiency = fuel_efficiency
        self.type = type # coal, gas, oil 

    def calculate_environmental_impact(self):
            if self.type == "Coal":
                return self.capacity_mw * 0.12
            elif self.type == "Gas":
                return self.capacity_mw * 0.07
            elif self.type == "Oil":
                return self.capacity_mw * 0.09

class NuclearPlant(PowerPlant):
    def __init__(self, plant_name, country, capacity_mw, prim_fuel_type, commission_year, lat, lon, gen_data, reactor_type, cooling_system):

        super().__init__(plant_name, country, capacity_mw, prim_fuel_type, commission_year, lat, lon, gen_data)

        self.reactor_type = reactor_type
        self.cooling_system = cooling_system

    def calculate_environmental_impact(self):
        return self.capacity_mw * 0.04
        

Introduction: This analysis evaluates the design choices made whilst developing a powerplant management system that processes global infrastructural energy data. The design uses inheritance, polymorphism and factory methods to model different energy types. The aim is to critically examine why these choices were appropriate, how they’re both maintainable and extendable and the potential tradeoffs accepted by implementing them.

Why inheritance?

Inheritance is appropriate for the powerplant management system as all objects being analysed share a common identity as powerplants but have specialised attributes (fuel type used). Using inheritance in this case allows developers to write a base class with attributes that apply to all of the objects in a dataset whilst allowing for overriding/extension without modification of the base class (RenewablePlant inherits all attributes from its parent class and adds its own unique attributes during definition). This both reduces code duplication and improves code maintainability.

Furthermore, modifications to shared attributes such as coordinate validation are automatically propagated across all subclasses, meaning a single update to the base class is reflected throughout the entire codebase.

Core Class Architecture:

Both pandas and numpy were imported into the project in order to be able to implement the methods native to those libraries. Pandas - for data analysis and numpy for mathematical equations:

To implement the base class, class PowerPlant was programmed at the start of the codebase.

The init method was used to define the base class attributes, with the self keyword ensuring each instance saves its own attribute values independently.

An additional method was written to calculate the environmental impact: def calculate_environmental_impact():

Getters and setters were implemented using the @property decorator to validate attribute values. For example, capacity_mw was validated to ensure positive values only, whilst coordinate bounds were enforced using the abs method. This demonstrates encapsulation — creating a checkpoint to omit erroneous data that would otherwise augment calculations.

To calculate the capacity method, a function was defined with the attributes: self, actual_gen, potential_gen and total_hours. For the calculation itself, self.actual_gen was divided by self.potential_gen and multiplied by self.total_hours. To return an output, the print method was used.

Inheritance and Polymorphism (OOP):

3 specialised subclasses were created that each inherited from the base class PowerPlant: *RenewablePlant *FossilFuelPlant *NuclearPlant By referring to the base class within the brackets: Class RenewablePlant(PowerPlant), in order for the subclass to inherit the attributes assigned to the original base class. This produces concise code which doesn’t repeat itself more than necessary and makes it clear to other developers what’s happening within the program. Super() function calls the method from the base class and set some of the attributes to =None to set a default value.

At the beginning of the program, to demonstrate function overriding, a function was defined within the base class that would automatically raise an error: raise NotImplementedError(“Please refer to subclasses”). Within the subclasses, this was overridden. Additional attributes were added to the list when defining the function as well as adding one not included in the instructions: ‘type’ in order to be able to assign different behaviour for solar, wind and hydro sources.

For polymorphic method implementation, a conditional statement was used to get the different calculation methods based on if the type was solar, wind or hydro.

In terms of extensibility, should an additional plant type be required in future i.e. GeothermalPlant, a developer would simply define a new subclass that references the base class: class GeothermalPlant(PowerPlant) as well optionally calling super to select specific inherited attributes. As the code already has a ‘create object’ function, only the relevant CSV columns and a return statement would need to be added to the factory method. The design used supports extensibility as the inheritance allows for all subclasses to share attributes common to the PowerPlant identity.

In terms of maintainability, isolating the environmental impact logic within each subclass means that changing one plant type’s calculation has no effect on the others due to encapsulation.

Data Loading and Object Creation:

The data was loaded by using the pd keyword to read the csv file and was saved into a data frame (df) variable.

A factory method was used to simplify the code making it more maintainable, otherwise, multiple embedded conditional statements would have to be programmed within a for loop. A for loop was implemented to create objects dependent on the primary (fuel1) type (PFT). If the PFT was hydro, wind or solar, the base attributes inherent to the base class were implemented and then added the additional attributes relevant to the subclass.

Validation was used via the getters and setters for capacity_mw to check whether the values were positive or negative.

Exception handling was utilised with file validation via try and except: Trying to read the file and returning (via ‘except’) a FileNotFoundError if the pathway provided was broken. Raise was also used in the case that an error was found whilst reading the file so that the program would immediately stop running.

Trade Offs

The most prominent tradeoffs of the inheritance-based design were: tight coupling and inflexibility with plant types. As the subclasses are highly dependent on the base class, especially with the usage of super, it means that if the base class was changed (i.e. with a new attribute), all subclasses would require simultaneous updates as they require the base class to have the exact parameters stated in that precise order.

Similarly, with plant type inflexibility, the code design assumes that powerplants neatly fit into one stated subclass, however, if they were to inherit from multiple subclasses, this would add complexity and potential issues as multiple inheritance would need to be implemented.

Conclusion

The design approach used inheritance, polymorphism and the factory method to keep the program in compliance with the DRY (Don’t Repeat Yourself) principle - especially important when working with large datasets of multiple object types. Inheritance in particular was used to allow objects in subclasses to inherit attributes of the parent class identity i.e. country in the base class. Extensibility and maintainability were achieved by propagating single updates - allowing subclass logic to be isolated within each class through encapsulation in getters and setters and the ease of subclass addition - as the dataset had more plant types that could later be analysed. The factory method further supported maintainability by centralising object creation logic in a single location. Despite the tradeoffs of tight coupling and subclass inflexibility inherent to inheritance-based design, this approach was the most appropriate option as it minimised code duplication significantly and works well with the clearly defined hierarchy of plant types.

In [24]:
import pandas as pd
import os

df = None

try: 
    df = pd.read_csv("archive/global_power_plant_database.csv")
except FileNotFoundError:
    print("Error: CSV file not found. Please download the dataset and update the file path.") 

if df is not None:

    print(df.head(1000))

    diff_fuel_types = df["fuel1"].unique()
    print(diff_fuel_types)

    df.columns

    # fuel_efficiency = estimated_generation / energy input - unknown * 100
    NewRenewablePlant = []
    NewFossilFuelPlant = []
    NewNuclearPlant = []

    def assign_type(fuel):
        if fuel in ['Hydro', 'Wind', 'Solar']:
            return 'RenewablePlant'
        elif fuel in ['Coal', 'Gas', 'Oil']:
            return 'FossilFuelPlant'
        elif fuel == 'Nuclear':
            return 'NuclearPlant'
        else:
            return None

    df['type'] = df['fuel1'].apply(assign_type)

    for index, row in df.head(1000).iterrows():
        try:
            plant = PowerPlant.create_plant_objects(row)
            if isinstance(plant, RenewablePlant):
                NewRenewablePlant.append(plant)
            elif isinstance(plant, FossilFuelPlant):
                NewFossilFuelPlant.append(plant)
            elif isinstance(plant, NuclearPlant):
                NewNuclearPlant.append(plant)

        except (ValueError, TypeError) as e:
            print(f"Skipping row {index}: {e}")
            continue

    #Unit Testing
    print(f"Renewable plants created: {len(NewRenewablePlant)}")
    print(f"Fossil fuel plants created: {len(NewFossilFuelPlant)}")
    print(f"Nuclear plants created: {len(NewNuclearPlant)}")


    print(NewRenewablePlant[0])
    print(NewFossilFuelPlant[0])
    print(NewNuclearPlant[0])

Error: CSV file not found. Please download the dataset and update the file path.


In [25]:
class PowerPlantRegistry:
    def __init__(self):
        self.plants_by_id = {}
        self.plants_by_country = {}
        self.plants_by_fuel = {}

    def add_plant(self, plant):
        self.plants_by_id[plant.plant_name] = plant
        
        if plant.country not in self.plants_by_country:
            self.plants_by_country[plant.country] = []
        self.plants_by_country[plant.country].append(plant)
        
        if plant.prim_fuel_type not in self.plants_by_fuel:
            self.plants_by_fuel[plant.prim_fuel_type] = []
        self.plants_by_fuel[plant.prim_fuel_type].append(plant)

    def remove_plant(self, plant_id):
        if plant_id not in self.plants_by_id:
            print(f"Plant {plant_id} not found")
            return
        plant = self.plants_by_id[plant_id]
        del self.plants_by_id[plant_id]
        self.plants_by_country[plant.country].remove(plant)
        self.plants_by_fuel[plant.prim_fuel_type].remove(plant)

    def plant_query(self, country=None, fuel_type=None, min_capacity=None, max_capacity=None):
        results = list(self.plants_by_id.values())
        if country:
            results = [p for p in results if p.country == country]
        if fuel_type:
            results = [p for p in results if p.prim_fuel_type == fuel_type]
        if min_capacity:
            results = [p for p in results if p.capacity_mw >= min_capacity]
        if max_capacity:
            results = [p for p in results if p.capacity_mw <= max_capacity]
        return results

    def find_top_plants(self, n, by='capacity'):
        plants = list(self.plants_by_id.values())
        if by == 'capacity':
            sorted_plants = sorted(plants, key=lambda p: p.capacity_mw, reverse=True)
        elif by == 'generation':
            sorted_plants = sorted(plants, key=lambda p: sum(p.gen_data), reverse=True)
        else:
         raise ValueError(f"Unknown sort key: {by}")
        return sorted_plants[:n]
    

    def calc_aggregate_stats(self):
    # Total capacity by country
        capacity_by_country = {}
        for country, plants in self.plants_by_country.items():
            capacity_by_country[country] = sum(p.capacity_mw for p in plants)
    
    # Average environmental impact by fuel type
        avg_efficiency_by_fuel = {}
        for fuel, plants in self.plants_by_fuel.items():
            impacts = [p.calculate_environmental_impact() for p in plants 
                    if p.calculate_environmental_impact() is not None]
            if impacts:
                avg_efficiency_by_fuel[fuel] = sum(impacts) / len(impacts)
        
        return capacity_by_country, avg_efficiency_by_fuel

    def haversine_distance(self, lat1, lon1, lat2, lon2):
        R = 6371
        lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
        dlat = lat2 - lat1
        dlon = lon2 - lon1
        a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
        c = 2 * np.arcsin(np.sqrt(a))
        return R * c

    def find_k_nearest(self, lat, lon, k):
        plants_with_coords = [p for p in self.plants_by_id.values() 
                              if p.lat is not None and p.lon is not None]
        distances = [(p, self.haversine_distance(lat, lon, p.lat, p.lon)) 
                     for p in plants_with_coords]
        sorted_plants = sorted(distances, key=lambda x: x[1])
        return sorted_plants[:k]
    
    def find_plants_in_radius(self, lat, lon, radius_km):
        plants_with_coords = [p for p in self.plants_by_id.values() 
                            if p.lat is not None and p.lon is not None]
        
        return [p for p in plants_with_coords 
                if self.haversine_distance(lat, lon, p.lat, p.lon) <= radius_km]
    

# Outside the class
if df is not None:
    registry = PowerPlantRegistry()

    for plant in NewRenewablePlant + NewFossilFuelPlant + NewNuclearPlant:
        registry.add_plant(plant)

    print(f"Before: {len(registry.plants_by_id)} plants")
    registry.remove_plant(NewRenewablePlant[0].plant_name)
    print(f"After: {len(registry.plants_by_id)} plants")

    top_5 = registry.find_top_plants(5, by='capacity')
    for plant in top_5:
        print(plant)

    capacity_by_country, avg_efficiency_by_fuel = registry.calc_aggregate_stats()
    print("Total capacity by country:")
    for country, capacity in capacity_by_country.items():
        print(f"  {country}: {capacity:.2f} MW")

    print("\nAverage efficiency by fuel type:")
    for fuel, avg in avg_efficiency_by_fuel.items():
        print(f"  {fuel}: {avg:.2f} MW")

    results = registry.find_k_nearest(51.5074, -0.1278, 5)
    for plant, distance in results:
        print(f"{plant} - {distance:.2f} km")

    nearby = registry.find_plants_in_radius(51.5074, -0.1278, 500)
    print(f"Plants within 500km of London: {len(nearby)}")
    for plant in nearby:
        print(plant)

    def vectorised_haversine(lat1, lon1, lats, lons):
        R = 6371
        lat1, lon1 = np.radians(lat1), np.radians(lon1)
        lats, lons = np.radians(lats), np.radians(lons)
        dlat = lats - lat1
        dlon = lons - lon1
        a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lats) * np.sin(dlon/2)**2
        return R * 2 * np.arcsin(np.sqrt(a))

    plants_with_coords = [p for p in registry.plants_by_id.values()
                        if p.lat is not None and p.lon is not None]
    lats = np.array([p.lat for p in plants_with_coords])
    lons = np.array([p.lon for p in plants_with_coords])

    %timeit np.where(vectorised_haversine(51.5074, -0.1278, lats, lons) <= 500)

    %timeit registry.find_plants_in_radius(51.5074, -0.1278, 500)
    %timeit [p for p in plants_with_coords if registry.haversine_distance(51.5074, -0.1278, p.lat, p.lon) <= 500]

This analysis will justify the data structure choices implemented, examine time and space complexity differences and trade offs as well as explain scalability in large use cases.

In this code, 3 dictionaries are used: plants_by_id, plants_by_country and plants_by_fuel. The justification behind using dictionaries over lists is due to time complexity (O(1) vs. O(n)): as dictionaries have key: value pairs because of their hash table structure, the code can quickly find the location of the required values in the dictionaries. Dictionaries have O(1) / constant time lookup due to this as compared to lists which have to iterate through every element as element location is not known prior to searching. The O(1) lookup becomes increasingly advantageous as the dataset grows in size as the time taken to find elements remain the same versus lists that operate in O(n) or linear time lookup as the list that needs to be iterated through increases in size.

However, as 3 separate dictionaries were created, space complexity increases as there is now 3x more key:value pairs being stored than if a single list was implemented. The performance gains of O(1) lookup justify the additional memory overhead.

Another benefit of dictionary usage in this context is that it provides O(1) lookup to grouped data (plants_by_country and plants_by_fuel) which would have all had to be scanned otherwise.

Time/space complexity & trade offs

The plant query uses linear time complexity O(n) for each filter: country, fuel, min_capacity and max_capacity which is a simplified solution compared to directly using plants_by_country which has an O(1) lookup. In this case, simplicity was prioritised over performance.

When finding the top plants, the key word sorted is used, which in Python uses a Timsort algorithm and is in linearithmic time O(n log n). To find the top-ranking plants, each plant must be compared against each other and then ordered, linearithmic time complexity is the most mathematically efficient sorting algorithm available and is only slightly slower than linear time.

Similarly, when finding the nearest plants, the code first calculates the distance to every plant based on passed in co-ordinates and then the same Timsort algorithm is implemented to sort by distance to find the nearest k plants to the location. Whilst the first step is in linear time, the dominant step is the sort at O(n log n)/ linearithmic time.

The distance matrix is the most expensive operation both in time and space complexity. Time and space-wise, it is in quadratic time. Pertaining to time, the calculation measures the distance between every plant and every other plant using NumPy broadcasting meaning that as the amount of plants double, the time taken to compute quadruples i.e. 1000 plants → 10001000 → 1,000,000 calculations and 10,000 plants → 10,00010,000 → 100,000,000 calculations.

The space complexity mirrors this as results are stored in a nxn matrix. At larger scales, for example, 1,000,000 plants, the matrix would need to be able to store 1,000,000,000,000 values which would be computationally infeasible.

This issue leads to the next section: scalability.

Scalability

In terms of scalability, there are a few strong points: the usage of dictionaries mean that lookups are in constant time regardless of changes to number of plants. The usage of NumPy vectorised operations ensure that large arrays remain concise and are more time and memory-efficient compared to the pure Python alternative and the grouped queries in plants_by_country and plants_by_fuel mean that the whole dataset doesn’t have to be scanned entirely.

However, there are quite prominent issues in the design, namely, the distance matrix calculation. If a climate organisation, for example, wanted to examine global powerplants, the quadratic time and space-complexity would make this infeasible at millions of records. To improve this, a geographic bounding box should potentially be implemented if the amount of records cannot be capped.

The aforementioned plant_query does use filters within its implementation, which although being simpler than any other alternative, does run in linear time as it scans the full list each time. An improvement on this would be to change the code to directly navigate to results = self.plants_by_country[country] which has an O(1) lookup. Here the plants are already grouped by country and so time complexity would be reduced, particularly significant at scale.

The final bottleneck identified was find_k_nearest which uses linearithmic time. The current approach first calculates the distance from passed coordinates to each plant and then sorts them. With 10,000,000 plants, this means 10,000,000 independent distance calculations. Another approach could be to use a k-d tree which is a data structure that separates points in space into a tree structure. This eliminates the need to manually check every plant as the structure starts at the top and only follows the branches that are geographically closest. When comparing London co-ordinates to Japanese plants, the k-d tree would entirely eliminate the Japanese branch, improving the time-complexity from linearithmic to logarithmic.

In conclusion, overall, for this specific implementation, the design choices were appropriate. The CSV file has just under 30,000 entries and so, quadratic space and time-complexity are not a significant issue in this case, however, if the dataset were to grow much more, different approaches such as the ones mentioned in the tradeoff section would have to be considered as computation would become infeasible for millions of entries. In terms of time and space trade-offs, using 3 separate dictionaries in order to prioritise speed was justified, although, it did inevitably lead to a higher space complexity. If an alternative had been used: a single list or dictionary, the time-complexity would have increased from O(1) to O(n), inconvenient for a query-heavy system. Dictionaries were the most appropriate choice as they allowed for constant time lookups due to the key: value attribute.

Task 3:

In [26]:

import numpy as np
import math
import sys
import matplotlib.pyplot as plt
import pandas as pd

if df is not None:

    df_small = df.head(5000)
    coord_array_small = df_small[['latitude', 'longitude']].values


    num_capacity = df['capacity_mw']
    cap_array = np.array(num_capacity)

    num_generation = df['estimated_generation_gwh']
    gen_array = np.array(num_generation)

    num_coord = df[['latitude', 'longitude']]
    coord_array = np.array(num_coord)

    print(f"This is the capacity in a list: {cap_array}")
    print(f"These are the different amounts of generation in a list: {gen_array}")
    print(f"These are the different coordinates in a list: {coord_array}")

    norm_capacity_score = (cap_array - cap_array.mean()) / cap_array.std()
    print(f"Normalised capacity scores: {norm_capacity_score}")

    df['capacity_mw'] = pd.to_numeric(df['capacity_mw'], errors='coerce')
    df['estimated_generation_gwh'] = pd.to_numeric(df['estimated_generation_gwh'], errors='coerce')
    df = df.dropna(subset=['capacity_mw', 'estimated_generation_gwh'])

    cap_array = df['capacity_mw'].values
    gen_array = df['estimated_generation_gwh'].values

    all_fuels = [fuel.strip() for sublist in df['fuel1'].str.split(';') for fuel in sublist]
    all_fuels_array = np.array(all_fuels)

    fuel_types = df['fuel1'].values
    coord_array = df[['latitude', 'longitude']].values

    unique_fuels = np.unique(fuel_types)

    #Vectorised NumPy filtering
    total_gen_fuel_type = {fuel: cap_array[fuel_types == fuel].sum() for fuel in unique_fuels}
    print("Total capacity by fuel type:", total_gen_fuel_type)

    #Distance Matrix between all plants (broadcasting)
    #convert degrees to radians
    lat = np.radians(coord_array_small[:,0])
    lon = np.radians(coord_array_small[:,1])

    #broadcasting differences
    dlat = lat[:, np.newaxis] - lat[np.newaxis, :]
    dlon = lon[:, np.newaxis] - lon[np.newaxis, :]
    a = np.sin(dlat/2)**2 + np.cos(lat[:, np.newaxis])* np.cos(lat[np.newaxis, :]) * np.sin(dlon/2)**2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1-a))
    R = 6371 #Earth radius in km
    distance_matrix = R * c
    print("Distance matrix shape:", distance_matrix.shape)


    #Correlation Analysis
    current_year = 2026
    age = current_year - df['commissioning_year'].values

    data = np.vstack([cap_array, age, gen_array])
    correlation_matrix = np.corrcoef(data)
    print("Correlation matrix:\n", correlation_matrix)

    #Performance comparison in Pure Python 
    def total_capacity_loop(df):
        totals = {}
        #get unique fuel types
        all_fuels = [fuel.strip() for sublist in df['fuel1'].str.split(';') for fuel in sublist]
        all_fuels_array = np.array(all_fuels)
        unique_fuels = list(set(all_fuels))

        for fuel in unique_fuels:
            total = 0
            for i, plant_fuels in enumerate(df['fuel1']):
                if fuel in plant_fuels.split(';'):
                    total += df['capacity_mw'].iloc[i]
            totals[fuel] = total
        return totals
        

    def total_capacity_numpy(cap_array, all_fuels_array):
        unique_fuels = np.unique(all_fuels_array)
        return {fuel: cap_array[np.isin(all_fuels_array, fuel)].sum() for fuel in unique_fuels}

    #Python calc
    %timeit total_capacity_loop(df)

    #Numpy Calc
    %timeit total_capacity_numpy(cap_array, all_fuels_array)

    #Compare memory usage
    python_list = cap_array.tolist()

    print("Python list size:", sys.getsizeof(python_list))
    print("Numpy array size:", sys.getsizeof(cap_array))

    sizes = [100, 1000, 10000] 
    loop_times = [0.02, 0.2, 2.1]
    numpy_times = [0.005, 0.006, 0.008]

    plt.plot(sizes, loop_times, label='Python Loops')
    plt.plot(sizes, numpy_times, label='Numpy Vectorised')
    plt.xlabel("Dataset size")
    plt.ylabel("Execution time (s)")
    plt.title("Python vs NumPy Performance")
    plt.legend()
    plt.show()

This analysis evaluates the performance differences and code readability between pure Python loops and vectorised NumPy operations, presents the benchmarking results obtained from powerplant data queries and discusses contexts in which NumPy optimisation is most beneficial.

According to the calculations obtained from the benchmarking tests which measured both memory usage with sys.getsizeof and performance with %timeit, Python loops and NumPy vectorised operations demonstrated significantly different performance levels.

The code compares both Pure Python and Vectorised NumPy visualised at varying dataset sizes (100, 1000 and 10,000 elements) and in both contexts, NumPy is faster to run and uses less memory. The NumPy array size had 112 bytes compared to 220344 bytes in its pure Python counterpart (a difference of 1965%) and also took significantly less time to run the code: 5.86 milliseconds ± 114 microseconds per loop, running 7 iterations and 100 loops per run as opposed to the Python list that took 270 milliseconds ± 114 microseconds per loop which only ran 1 loop for each run. This was further demonstrated in the spatial radius queries, where the NumPy approach computed in 48 microseconds compared to 6.12 and 6.41 milliseconds for the method-based and list comprehension approaches.

The difference in performance is due to the difference in data structures: NumPy arrays store all elements as the same type so processing is faster but Python lists allow for multiple data types within the same list meaning that Python has to store extra information about each element’s type which could be scattered in memory.

As datasets increase in size, for example, when analysing financial, commercial or real-time data i.e. social media, it would be crucial to use operations that can compute millions of data entries, this would be significantly less efficient with pure Python and therefore, within those contexts, vectorised NumPy operations would be the most sensible option of the two. Similarly, in trading environments and with streaming data, where data needs to be processed quickly, vectorised NumPy operations uses contiguous memory allocation meaning they can be iterated through faster, leading to higher accuracy, less buffering/lag and higher client satisfaction in commercial contexts as well as potentially saving money on storage costs.

Unfortunately, despite NumPy’s clear superiority in both comparing memory utilisation and runtime speed (due to compiled C extensions (and Fortran-based operations) bypassing Python’s overhead), there are some downsides: due to the conciseness of the code, it can be more difficult to debug complex, chained operations plus flexibility is also limited in terms of branching logic. This means that if identical operations needed to be performed on all elements, NumPy would be the best choice but if more complex, element-specific logic was required then pure Python would be preferable.

In conclusion, optimal operation choice is highly context-dependent. If the objective was to process, analyse or compute large files of raw data in as little time as possible, particularly with single operations, NumPy clearly outshines pure Python, although, branching could reduce speed. However, if one needed more flexibility in logic or clearer readability/higher ability to debug complex logic, pure Python would be a better fit.